# Classification Review

Displays `base_classifications.csv` alongside the original benchmark entry and ground-truth answer for each row, so you can spot-check the model's labels.

**Controls (Cell 2):**
- Set `CATEGORY` to the benchmark folder name.
- Use `FILTER_*` options to narrow down what you see.
- Run Cell 3 to load data, then Cell 4 to browse.

In [1]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import csv
import json
from pathlib import Path
from IPython.display import display, HTML

NOTEBOOK_DIR = Path().resolve()
PACKAGE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_ROOT    = PACKAGE_ROOT / "data" / "benchmarks"

print(f"Package root: {PACKAGE_ROOT}")

Package root: C:\Users\omnoy\Documents\BIU\Thesis\multilingual-tool-use-evaluation\multilingual-bfcl


In [7]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
CATEGORY = "multiple"   # benchmark folder under data/benchmarks/

# ── Filters (set to None to disable) ─────────────────────────────────────────
# Show only entries where a classification column equals a specific value:
FILTER_parameter_type        = None   # "universal" | "textual" | "mixed" | None
FILTER_convertible_units     = None   # "true" | "false" | None
FILTER_localizable_query     = None   # "true" | "false" | None
FILTER_localizable_parameters= None   # "true" | "false" | None

# Show only entries whose ID contains this string (e.g. "multiple_42"), or None:
FILTER_id = None

# Maximum number of entries to display (None = all):
MAX_DISPLAY = None

In [8]:
# ── Cell 3: Load data ─────────────────────────────────────────────────────────
bench_dir   = DATA_ROOT / CATEGORY
source_path = bench_dir / "eng_base.json"
answer_path = bench_dir / "possible_answer" / "eng_base.json"
csv_path    = bench_dir / "base_classifications.csv"

for p in (source_path, answer_path, csv_path):
    if not p.exists():
        raise FileNotFoundError(f"Missing: {p}")

def load_jsonl(path):
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

entries = {e["id"]: e for e in load_jsonl(source_path)}
answers = {a["id"]: a for a in load_jsonl(answer_path)}

with open(csv_path, newline="", encoding="utf-8") as f:
    classifications = list(csv.DictReader(f))

print(f"Entries  : {len(entries)}")
print(f"Answers  : {len(answers)}")
print(f"Classified: {len(classifications)}")

# Value distribution
from collections import Counter
for col in ("parameter_type", "localizable_query", "localizable_parameters"):
    counts = Counter(r[col] for r in classifications)
    print(f"  {col}: {dict(counts)}")

Entries  : 200
Answers  : 200
Classified: 200
  parameter_type: {'non-translatable': 84, 'translatable': 116}
  localizable_query: {'': 84, 'true': 95, 'false': 21}
  localizable_parameters: {'': 84, 'true': 94, 'false': 22}


In [9]:
# ── Cell 4: Browse ────────────────────────────────────────────────────────────

LABEL_COLORS = {
    "translatable": ("#1a3a5c", "#7ec8f7"),
    "non-translatable":   ("#4a2800", "#ffb347"),
    "true":      ("#0d3320", "#5fd98a"),
    "false":     ("#3a0d0d", "#f77a7a"),
    "error":       ("#3a0d0d", "#f77a7a"),
    "parse_error": ("#3a0d0d", "#f77a7a"),
}

CARD_BG     = "#1e1e1e"
CARD_BORDER = "#3a3a3a"
SECTION_BG  = "#2a2a2a"
TITLE_COLOR = "#e0e0e0"
LABEL_COLOR = "#aaaaaa"
TEXT_COLOR  = "#d4d4d4"
USER_COLOR  = "#7eb8f7"
ASST_COLOR  = "#aaaaaa"

def resolve_id(csv_id: str):
    """Map a CSV id to the matching entry/answer key.

    langasync submits batches with sequential integer indices (0, 1, 2, …)
    instead of our entry IDs.  We try three strategies:
      1. Direct match (csv_id == entry key)          e.g. "multiple_0"
      2. Category-prefixed  (CATEGORY + "_" + csv_id) e.g. "multiple_0"
      3. Numeric index into the sorted entry list
    """
    if csv_id in entries:
        return csv_id
    prefixed = f"{CATEGORY}_{csv_id}"
    if prefixed in entries:
        return prefixed
    # Fall back to positional index if csv_id is a plain integer
    try:
        idx = int(csv_id)
        key = sorted(entries.keys())[idx]
        return key
    except (ValueError, IndexError):
        return None

def badge(value: str) -> str:
    bg, fg = LABEL_COLORS.get(value.lower(), ("#333", "#ccc"))
    return (f'<span style="background:{bg};color:{fg};border-radius:4px;'
            f'padding:2px 8px;font-weight:600;font-size:0.85em">{value}</span>')

def format_ground_truth(ground_truth):
    lines = []
    for call in ground_truth:
        for func_name, params in call.items():
            lines.append(f'<b style="color:{TITLE_COLOR}">{func_name}</b>')
            for param, values in params.items():
                display_val = next(
                    (v for v in values if v != "" and v is not None),
                    values[0] if values else ""
                )
                lines.append(
                    f'&nbsp;&nbsp;<code style="color:#ce9178">{param}</code>'
                    f' = <code style="color:#b5cea8">{json.dumps(display_val, ensure_ascii=False)}</code>'
                )
    return "<br>".join(lines)

def format_query(entry):
    parts = []
    for turn in entry.get("question", []):
        for msg in turn:
            role  = msg.get("role", "?")
            color = USER_COLOR if role == "user" else ASST_COLOR
            parts.append(
                f'<span style="color:{color};font-weight:600">{role}:</span> '
                f'<span style="color:{TEXT_COLOR}">{msg["content"]}</span>'
            )
    return "<br>".join(parts)

def format_functions(entry):
    lines = []
    for func in entry.get("function", []):
        name   = func.get("name", "?")
        params = func.get("parameters", {}).get("properties", {})
        req    = set(func.get("parameters", {}).get("required", []))
        param_strs = []
        for p, pdef in params.items():
            star = "*" if p in req else ""
            param_strs.append(
                f'<code style="color:#ce9178">{p}{star}</code>'
                f'<span style="color:#888">: {pdef.get("type","any")}</span>'
            )
        lines.append(f'<b style="color:#dcdcaa">{name}</b>({", ".join(param_strs)})')
    return "<br>".join(lines)

def render_entry(clf):
    csv_id   = clf["id"]
    real_id  = resolve_id(csv_id)
    entry    = entries.get(real_id)  if real_id else None
    answer   = answers.get(real_id) if real_id else None
    label    = real_id or csv_id

    gt_html    = format_ground_truth(answer["ground_truth"]) if answer else "<i style='color:#888'>no answer</i>"
    query_html = format_query(entry)     if entry else "<i style='color:#888'>no entry</i>"
    func_html  = format_functions(entry) if entry else "<i style='color:#888'>—</i>"

    return f"""
<div style="border:1px solid {CARD_BORDER};border-radius:8px;margin:12px 0;padding:16px;
            font-family:sans-serif;background:{CARD_BG}">
  <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:10px">
    <span style="font-size:1.05em;font-weight:700;color:{TITLE_COLOR}">{label}</span>
    <div style="display:flex;gap:6px;flex-wrap:wrap;justify-content:flex-end">
      {badge(clf['parameter_type'])}
      <span style="color:{LABEL_COLOR};font-size:0.85em">local. query:&nbsp;{badge(clf['localizable_query'])}</span>
      <span style="color:{LABEL_COLOR};font-size:0.85em">local. params:&nbsp;{badge(clf['localizable_parameters'])}</span>
    </div>
  </div>
  <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;font-size:0.9em">
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">QUERY</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.6">{query_html}</div>
    </div>
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">FUNCTIONS</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.8">{func_html}</div>
    </div>
    <div>
      <div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">GROUND TRUTH</div>
      <div style="background:{SECTION_BG};padding:8px;border-radius:4px;line-height:1.8">{gt_html}</div>
    </div>
  </div>
</div>"""

# ── Apply filters ─────────────────────────────────────────────────────────────
filtered = classifications
for col, val in [
    ("parameter_type",         FILTER_parameter_type),
    ("convertible_units",      FILTER_convertible_units),
    ("localizable_query",      FILTER_localizable_query),
    ("localizable_parameters", FILTER_localizable_parameters),
]:
    if val is not None:
        filtered = [r for r in filtered if r[col] == val]
if FILTER_id is not None:
    filtered = [r for r in filtered if FILTER_id in r["id"]]

shown = filtered[:MAX_DISPLAY] if MAX_DISPLAY else filtered
print(f"Showing {len(shown)} of {len(filtered)} matching entries "
      f"(total classified: {len(classifications)})")

html = "".join(render_entry(r) for r in shown)
display(HTML(html))

Showing 200 of 200 matching entries (total classified: 200)
